In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings as wr

df = pd.read_csv('./styles.csv', on_bad_lines='skip')
df.head()
print("hi")





hi


In [4]:
df.dtypes

id                      int64
gender                 object
masterCategory         object
subCategory            object
articleType            object
baseColour             object
season                 object
year                  float64
usage                  object
productDisplayName     object
dtype: object

In [5]:
obj_df = df.select_dtypes(include=['object']).copy()
obj_df.head()

,gender,masterCategory,subCategory,articleType,baseColour,season,usage,productDisplayName
0,Men,Apparel,Topwear,Shirts,Navy Blue,Fall,Casual,Turtle Check Men Navy Blue Shirt
1,Men,Apparel,Bottomwear,Jeans,Blue,Summer,Casual,Peter England Men Party Blue Jeans
2,Women,Accessories,Watches,Watches,Silver,Winter,Casual,Titan Women Silver Watch
3,Men,Apparel,Bottomwear,Track Pants,Black,Fall,Casual,Manchester United Men Solid Black Track Pants
4,Men,Apparel,Topwear,Tshirts,Grey,Summer,Casual,Puma Men Grey T-shirt


In [6]:
obj_df[obj_df.isnull().any(axis=1)]

,gender,masterCategory,subCategory,articleType,baseColour,season,usage,productDisplayName
87,Women,Personal Care,Nails,Nail Polish,Bronze,Spring,NaN,Streetwear Ash Nail Polish # 31
92,Unisex,Apparel,Topwear,Rain Jacket,Coffee Brown,Summer,NaN,Just Natural Unisex Charcoal Rain Jacket
282,Women,Footwear,Shoes,Sports Shoes,Purple,NaN,Sports,Kalenji Ekiden 200 Wn Purple 2011
292,Women,Personal Care,Lips,Lipstick,Pink,Spring,NaN,Lakme Absolute Lip Last Day Kiss Lip Colour
479,Women,Personal Care,Lips,Lipstick,Brown,Spring,NaN,Lotus Herbals Pure Colours Nutty Brown Lipstic...
...,...,...,...,...,...,...,...,...
43633,Women,Personal Care,Makeup,Kajal and Eyeliner,Black,Spring,NaN,Streetwear Black Eye Liner 01
44079,Women,Personal Care,Lips,Lip Gloss,Red,Spring,NaN,Lotus Herbals Seduction Sappy Watermelon Lip G...
44224,Men,Personal Care,Fragrance,Perfume and Body Mist,NaN,Spring,NaN,GUESS by Marciano Men Eau De Toilette 50 ml
44227,Women,Personal Care,Lips,Lipstick,Purple,Spring,NaN,Lakme Enrich Satins Lipstick 461


In [7]:
obj_df["usage"].value_counts()

usage
Casual          34406
Sports           4025
Ethnic           3208
Formal           2345
Smart Casual       67
Party              29
Travel             26
Home                1
Name: count, dtype: int64

In [8]:
df = df.dropna()
df.isnull().sum()

id                    0
gender                0
masterCategory        0
subCategory           0
articleType           0
baseColour            0
season                0
year                  0
usage                 0
productDisplayName    0
dtype: int64

In [9]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, ConfusionMatrixDisplay
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from scipy.stats import randint

df.get_dummies(df, dtype = float)

In [10]:
df.dtypes

id                      int64
gender                 object
masterCategory         object
subCategory            object
articleType            object
baseColour             object
season                 object
year                  float64
usage                  object
productDisplayName     object
dtype: object

In [11]:
#this code encodes the categorical data in our dataframe into numerical values so we can input into random forest succesfully
from sklearn.preprocessing import LabelEncoder
label_encoder = LabelEncoder()
#this for loop runs through each column individually to convert to numbers
for column in df.select_dtypes(include=['object']).columns:
    df[column] = label_encoder.fit_transform(df[column])



In [12]:
#X is all the columns except color and y is color
#model is predicting color
X = df.drop(['baseColour', 'id', 'gender'], axis=1)
y = df['baseColour']

#Split training and testing data for X and y
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

#feed into data model - using random forest
#this creates a trained random forest model
rf = RandomForestClassifier()
rf.fit(X_train, y_train)

#this looks at if the previous model is making accurate predictions
y_pred = rf.predict(X_test)

#this creates an accuracy percentage
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy: ", accuracy)
#look into random forest hyperparamters



Accuracy:  0.4922867513611615


In [13]:
#X is all the columns except color and y is color
#model is predicting color
X = df.drop(['articleType','id'], axis=1)
y = df['articleType']

#Split training and testing data for X and y
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

#feed into data model - using random forest
#this creates a trained random forest model
rf = RandomForestClassifier()
rf.fit(X_train, y_train)

#this looks at if the previous model is making accurate predictions
y_pred = rf.predict(X_test)

#this creates an accuracy percentage
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy: ", accuracy)

Accuracy:  0.8430127041742287


In [19]:
# Train a new model with only the 'year' column as the feature
X_year = df[['year', 'season', 'gender', 'subCategory']]  # Only using 'year' as the feature
y = df['articleType']   # Target remains 'articleType'

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X_year, y, test_size=0.2)

# Train the RandomForest model with only the year feature
rf_year = RandomForestClassifier()
rf_year.fit(X_train, y_train)

# Check accuracy
y_pred = rf_year.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy with only 'year':", accuracy)

Accuracy with only 'year': 0.6555127041742287


In [ ]:
def predict_article_type_by_year(year, gender, seadon, subCategory, model):
    """
    Predict the article type based on the year using the trained model.
    
    Parameters:
    - year (float or int): The year for which to predict the article type.
    - model: The trained RandomForestClassifier model.
    
    Returns:
    - Predicted article type.
    """
    # Format the year as required by the model
    year_input = [[year]]  # Model expects a 2D array for a single input
    
    # Predict the article type
    article_type_prediction = model.predict(year_input)[0]
    
    return article_type_prediction

# Example usage
predicted_article_type = predict_article_type_by_year(2022, rf_year)
print("Predicted Article Type for year 2022:", predicted_article_type)

In [15]:
#X is all the columns except color and y is color
#model is predicting color
X = df.drop(['subCategory','id'], axis=1)
y = df['subCategory']

#Split training and testing data for X and y
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

#feed into data model - using random forest
#this creates a trained random forest model
rf = RandomForestClassifier()
rf.fit(X_train, y_train)

#this looks at if the previous model is making accurate predictions
y_pred = rf.predict(X_test)

#this creates an accuracy percentage
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy: ", accuracy)

Accuracy:  0.9924001814882033


In [16]:
from flask import Flask, request, jsonify
from flask_cors import CORS
import pandas as pd
import pickle
from sklearn.ensemble import RandomForestClassifier

# Load your model and data pre-processing steps here
app = Flask(__name__)
CORS(app)  # To enable cross-origin requests

# Initialize or load the trained model
rf = RandomForestClassifier()
# Ensure to replace with the pre-trained model
# rf.fit(X_train, y_train) # Uncomment if training here

# Prediction endpoint
@app.route('/predict', methods=['POST'])
def predict():
    data = request.get_json()
    # Convert incoming data to DataFrame, if required
    X_new = pd.DataFrame(data)
    prediction = rf.predict(X_new)
    return jsonify(prediction.tolist())

if __name__ == "__main__":
    app.run(debug=True)


 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
 * Restarting with stat
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Users/aparn/Library/Python/3.11/lib/python/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/Users/aparn/Library/Python/3.11/lib/python/site-packages/traitlets/config/application.py", line 1074, in launch_instance
    app.initialize(argv)
  File "/Users/aparn/Library/Python/3.11/lib/python/site-packages/traitlets/config/application.py", line 118, in inner
    return method(app, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/aparn/Library/Python/3.11/lib/python/site-packages/ipykernel/kernelapp.py", line 692, in initialize
    self.init_sockets()
  File "/Users/aparn/Library/Python/3.11/lib/python/site-packages/ipykernel/kernelapp.py", line 331, in init_sockets
    self.shell_port = self._

SystemExit: 1

/Users/aparn/Library/Python/3.11/lib/python/site-packages/IPython/core/interactiveshell.py:3585: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [69]:
#X is all the columns except color and y is color
#model is predicting color
X = df.drop(['productDisplayName','id'], axis=1)
y = df['productDisplayName']

#Split training and testing data for X and y
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

#feed into data model - using random forest
#this creates a trained random forest model
rf = RandomForestClassifier()
rf.fit(X_train, y_train)

#this looks at if the previous model is making accurate predictions
y_pred = rf.predict(X_test)

#this creates an accuracy percentage
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy: ", accuracy)

: 

In [67]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

# Define the parameter grid
param_grid = {
    'n_estimators': [100, 200, 300],  # try a range from 100 to 300
    'max_depth': [None, 10, 20, 30],  # try different max depths
    'min_samples_split': [2, 5, 10],  # options for minimum samples to split
    'min_samples_leaf': [1, 2, 4],    # options for minimum samples in a leaf node
    'max_features': ['sqrt', 'log2']  # try sqrt or log2 for feature sampling
}

# Set up GridSearchCV with 5-fold cross-validation
grid_search = GridSearchCV(estimator=RandomForestClassifier(), param_grid=param_grid, cv=5, n_jobs=-1, verbose=2)

# Fit the grid search to the data
grid_search.fit(X_train, y_train)

# Get the best model
best_rf = grid_search.best_estimator_

# Test the model on the test set
y_pred = best_rf.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print("Best Hyperparameters: ", grid_search.best_params_)
print("Accuracy with optimized hyperparameters: ", accuracy)


Fitting 5 folds for each of 216 candidates, totalling 1080 fits


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/model_selection/_split.py:776: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


KeyboardInterrupt: 

In [20]:
from sklearn.model_selection import RandomizedSearchCV

# Define parameter distributions for randomized search
param_dist = {
    'n_estimators': [50, 100, 200],  # reduce the range of estimators
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2']
}

# Set up RandomizedSearchCV with 20 iterations
random_search = RandomizedSearchCV(estimator=RandomForestClassifier(),
                                   param_distributions=param_dist,
                                   n_iter=20,  # number of random combinations to try
                                   cv=5, n_jobs=-1, verbose=2, random_state=42)
random_search.fit(X_train, y_train)


Fitting 5 folds for each of 20 candidates, totalling 100 fits


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/model_selection/_split.py:776: UserWarning: The least populated class in y has only 4 members, which is less than n_splits=5.
  warnings.warn(


[CV] END max_depth=20, max_features=sqrt, min_samples_leaf=1, min_samples_split=2, n_estimators=100; total time=   3.3s
[CV] END max_depth=20, max_features=sqrt, min_samples_leaf=1, min_samples_split=2, n_estimators=100; total time=   3.3s
[CV] END max_depth=20, max_features=sqrt, min_samples_leaf=1, min_samples_split=2, n_estimators=100; total time=   3.5s
[CV] END max_depth=20, max_features=log2, min_samples_leaf=4, min_samples_split=5, n_estimators=200; total time=   4.5s
[CV] END max_depth=20, max_features=log2, min_samples_leaf=4, min_samples_split=5, n_estimators=200; total time=   4.6s
[CV] END max_depth=20, max_features=log2, min_samples_leaf=4, min_samples_split=5, n_estimators=200; total time=   4.7s
[CV] END max_depth=20, max_features=log2, min_samples_leaf=4, min_samples_split=5, n_estimators=200; total time=   4.7s
[CV] END max_depth=20, max_features=log2, min_samples_leaf=4, min_samples_split=5, n_estimators=200; total time=   4.7s
[CV] END max_depth=10, max_features=sqrt

RandomizedSearchCV(cv=5, estimator=RandomForestClassifier(), n_iter=20,
                   n_jobs=-1,
                   param_distributions={'max_depth': [None, 10, 20],
                                        'max_features': ['sqrt', 'log2'],
                                        'min_samples_leaf': [1, 2, 4],
                                        'min_samples_split': [2, 5, 10],
                                        'n_estimators': [50, 100, 200]},
                   random_state=42, verbose=2)

In [21]:
# Initialize RandomForest with optimized parameters
rf_optimized = RandomForestClassifier(max_features='log2', n_estimators=200)

# Train the model with the optimized parameters
rf_optimized.fit(X_train, y_train)

# Make predictions and calculate accuracy
y_pred_optimized = rf_optimized.predict(X_test)
accuracy_optimized = accuracy_score(y_test, y_pred_optimized)

print("Optimized Model Accuracy: ", accuracy_optimized)


Optimized Model Accuracy:  0.49432849364791287


In [22]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Prepare X and y
X = df.drop(['baseColour', 'id', 'gender'], axis=1)
y = df['baseColour']

# Split training and testing data for X and y
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# Initialize the XGBoost classifier
xgb_model = xgb.XGBClassifier()

# Train the XGBoost model
xgb_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = xgb_model.predict(X_test)

# Calculate and print the accuracy
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy: ", accuracy)


Accuracy:  0.3242967332123412


In [23]:
import platform
print(platform.architecture())

('64bit', '')


In [59]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import accuracy_score
from tensorflow.keras.layers import Dropout

# Prepare data
X = df.drop(['baseColour', 'id', 'gender'], axis=1)
y = df['baseColour']

# Encode target labels if needed (assuming baseColour is categorical)
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
y_categorical = to_categorical(y_encoded)  # One-hot encode

# Split training and testing data for X and y
X_train, X_test, y_train, y_test = train_test_split(X, y_categorical, test_size=0.2)

# Standardize the features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Build neural network model
model = Sequential()
model.add(Dense(512, activation='relu', input_shape=(X_train.shape[1],)))
model.add(Dropout(0.4))
model.add(Dense(256, activation='relu'))
model.add(Dropout(0.3))
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.3))
model.add(Dense(y_categorical.shape[1], activation='softmax'))  # Output layer for classification

# Compile the model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Train the neural network model
model.fit(X_train, y_train, epochs=100, batch_size=32, validation_split=0.2)

# Make predictions on the test set
y_pred_proba = model.predict(X_test)
y_pred = y_pred_proba.argmax(axis=1)
y_test_labels = y_test.argmax(axis=1)

# Calculate and print the accuracy
accuracy = accuracy_score(y_test_labels, y_pred)
print("Accuracy: ", accuracy)


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/100
882/882 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.2083 - loss: 2.8205 - val_accuracy: 0.2390 - val_loss: 2.5743
Epoch 2/100
882/882 ━━━━━━━━━━━━━━━━━━━━ 1s 988us/step - accuracy: 0.2393 - loss: 2.5901 - val_accuracy: 0.2514 - val_loss: 2.4918
Epoch 3/100
882/882 ━━━━━━━━━━━━━━━━━━━━ 1s 976us/step - accuracy: 0.2536 - loss: 2.5304 - val_accuracy: 0.2514 - val_loss: 2.4739
Epoch 4/100
882/882 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.2527 - loss: 2.4972 - val_accuracy: 0.2593 - val_loss: 2.4583
Epoch 5/100
882/882 ━━━━━━━━━━━━━━━━━━━━ 1s 985us/step - accuracy: 0.2615 - loss: 2.4702 - val_accuracy: 0.2600 - val_loss: 2.4444
Epoch 6/100
882/882 ━━━━━━━━━━━━━━━━━━━━ 1s 993us/step - accuracy: 0.2586 - loss: 2.4605 - val_accuracy: 0.2578 - val_loss: 2.4279
Epoch 7/100
882/882 ━━━━━━━━━━━━━━━━━━━━ 1s 985us/step - accuracy: 0.2637 - loss: 2.4475 - val_accuracy: 0.2562 - val_loss: 2.4272
Epoch 8/100
882/882 ━━━━━━━━━━━━━━━━━━━━ 1s 992us/step - accuracy: 0.2636 - loss: 2.441

In [26]:
!pip3 install tensorflow


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 239.5/239.5 MB 54.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 90.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 51.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 85.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 79.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 77.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 72.8 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.1.2
    Uninstalling numpy-2.1.2:
      Successfully uninstalled numpy-2.1.2

[notice] A new release of pip is available: 24.2 -> 24.3.1
[notice] To update, run: python3.11 -m pip install --upgrade pip


In [54]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score


# Prepare data
X = df.drop(['baseColour', 'id', 'gender'], axis=1)
y = df['baseColour']

# Encode target labels if needed (assuming baseColour is categorical)
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# Split training and testing data for X and y
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)

# Standardize the features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Initialize the K-Nearest Neighbors classifier
knn_model = KNeighborsClassifier(n_neighbors=18, weights='distance', metric='minkowski', p=6)  # You can try different values for n_neighbors

# Train the KNN model
knn_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = knn_model.predict(X_test)

# Calculate and print the accuracy
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy: ", accuracy)


Accuracy:  0.5001134301270418


In [38]:
from sklearn.model_selection import GridSearchCV

param_grid = {'n_neighbors': range(1, 20)}
grid_search = GridSearchCV(KNeighborsClassifier(weights='distance'), param_grid, cv=5)
grid_search.fit(X_train, y_train)

print("Best n_neighbors:", grid_search.best_params_)
print("Best accuracy with optimal n_neighbors:", grid_search.best_score_)


Best n_neighbors: {'n_neighbors': 18}
Best accuracy with optimal n_neighbors: 0.4847565097227145


In [52]:
from sklearn.model_selection import cross_val_score

knn_model = KNeighborsClassifier(n_neighbors=5, weights='distance')
scores = cross_val_score(knn_model, X_train, y_train, cv=5)
print("Cross-validation scores:", scores)
print("Mean cross-validation accuracy:", scores.mean())


Cross-validation scores: [0.48192259 0.47702779 0.48128191 0.48681225 0.4771696 ]
Mean cross-validation accuracy: 0.48084282692608815


In [55]:
from sklearn.svm import SVC

svm_model = SVC(kernel='rbf', C=1.0, gamma='scale')  # Use 'linear' kernel for a linear SVM
svm_model.fit(X_train, y_train)
y_pred = svm_model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print("SVM Accuracy:", accuracy)


SVM Accuracy: 0.2658802177858439


In [56]:
from sklearn.linear_model import LogisticRegression

logreg_model = LogisticRegression(max_iter=200, random_state=42)
logreg_model.fit(X_train, y_train)
y_pred = logreg_model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print("Logistic Regression Accuracy:", accuracy)


Logistic Regression Accuracy: 0.22039473684210525


In [57]:
print(df['baseColour'].value_counts())


baseColour
1     9700
44    5497
2     4907
4     3440
13    2735
33    2434
12    2103
31    1824
25    1784
32    1612
37    1089
45     776
0      745
11     621
19     577
29     523
28     409
22     394
9      384
39     315
6      228
30     185
27     182
38     163
16     160
14     146
15     139
18     128
42     119
40     112
24      97
3       89
8       83
43      69
35      65
5       44
21      41
7       29
20      28
36      22
34      21
26      21
23      16
41      11
17       5
10       5
Name: count, dtype: int64


In [60]:
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)

importances = rf_model.feature_importances_
sorted_indices = np.argsort(importances)[::-1]

# Print the most important features
for i in sorted_indices[:10]:
    print(f"{X.columns[i]}: {importances[i]}")


productDisplayName: 0.8651717549727603
articleType: 0.04387232963570702
subCategory: 0.026261357196393095
year: 0.02434460271822209
season: 0.015544753546281981
usage: 0.01375832581600218
masterCategory: 0.011046876114633297
